In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from benchmark import ModelBenchmark
from helpers import DATA_PATH, get_data_from_file

data folder is set to `c:\users\brumda\documents\neuspell\neuspell\../neuspell_data` script


In [3]:
corrupt, clean = get_data_from_file('small')

In [4]:
len(corrupt)

67886

### Baseline

In [203]:
from symspellpy import symspellpy
import pkg_resources

max_edit_distance = 2
prefix_length = 7

sym_spell = symspellpy.SymSpell(max_edit_distance, prefix_length)
dictionary_path = pkg_resources.resource_filename(
        "symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)
# bigram_path = pkg_resources.resource_filename(
#     "symspellpy", "frequency_bigramdictionary_en_243_342.txt")
# sym_spell.load_bigram_dictionary(bigram_path, term_index=0, count_index=2)

sym_spell.create_dictionary(DATA_PATH + "corpus.txt", encoding='utf-8')
input_phrase = corrupt[0]
suggestions = sym_spell.lookup_compound(input_phrase, max_edit_distance=max_edit_distance)



In [211]:
print(suggestions[0].term)

team number tr 1 p span contents 0


In [27]:
benchmark = ModelBenchmark(device='cpu')

Model loaded from pred_typo_models/best_model.pt


In [ ]:
benchmark.benchmark_model(sym_spell,
                          clean,
                          corrupt,
                          "symspell",
                          lambda model, data: model.correct_string(data),
                          warm_up_runs=0,
                          num_runs=2)


### Neuspell


In [59]:

from neuspell import BertChecker

checker = BertChecker(device='cuda')
checker.from_pretrained()

loading vocab from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise\vocab.pkl
initializing model
loading pretrained weights from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise
Loading model params from checkpoint dir: c:\users\brumda\documents\neuspell\neuspell\../neuspell_data/checkpoints/subwordbert-probwordnoise


In [62]:
print(checker.correct_string('very skeletal "app" definition, just to show the basic usage.', correct_spaces=False))

very skeletal " app " definition , just to show the basic usage .


#### Bench

### FIX

In [91]:
def get_subtokens(tokens, start_index):
    """Get the first word after start index"""
    result = [tokens[start_index]]

    # Find indices of tokens after start_index that start with '##'
    mask = np.char.startswith(tokens[start_index + 1:], '##')

    # Find the first False (non-## token) in the mask
    # Else happens if the rest of the tokens are one word
    non_subtoken_indices = np.where(~mask)[0]
    end_idx = non_subtoken_indices[0] if len(non_subtoken_indices) > 0 else len(mask)

    result.extend(tokens[start_index + 1:start_index + 1 + end_idx])
    return np.array(result)

In [96]:
from transformers import BertTokenizerFast
import numpy as np

index = 378
tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
tokenizer.do_basic_tokenize = True
tokenizer.tokenize_chinese_chars = False
orig_string = corrupt[index]
orig_string_clean = clean[index]
# test_string = '### General Contributition Guidelines'
# test_string_clean = '### General Contributition Guidelines'
pred_string = checker.correct_string(orig_string, correct_spaces=False)
# transformed_tokens = transformed_tokens[:14] + 's' + transformed_tokens[14:29] + 's' + transformed_tokens[29:]
print(f"input: {orig_string}", f"clean: {orig_string_clean}", f"output: {pred_string}", sep="\n")

input: ### General Contributition Guidelines:
clean: ### General Contribution Guidelines:
output: # # # General Contribution Guidelines :


In [97]:
original_offsets = tokenizer(orig_string, return_offsets_mapping=True)['offset_mapping']
tokenization = np.array(tokenizer.tokenize(pred_string))
token_offsets = np.array(original_offsets[1:-1])  # first and last offsets are [CLS] and [SEP]
idx_offset = 0
out_idx = 0
in_row = 0

# Pre-allocate arrays
in_tok = np.array(tokenizer.tokenize(orig_string))
max_tokens = max(len(tokenization), len(original_offsets))
pretok_sent = np.empty(max_tokens, dtype=object)
offsets_merged = np.empty((max_tokens, 2), dtype=int)

i = 0
delta = 0
old_offset = 0
while i < len(tokenization):
    token = tokenization[i]
    in_bounds = i + idx_offset < len(token_offsets)
    if token.startswith("##"):
        # Continuation of previous token
        in_row += 1
        # merge
        pretok_sent[out_idx - 1] = pretok_sent[out_idx - 1] + token[2:]
        if in_bounds:
            old_offset = offsets_merged[out_idx - 1, 1]
            offsets_merged[out_idx - 1, 1] = token_offsets[i + idx_offset, 1]
            # print(old_offset, offsets_merged[out_idx - 1])
        else:
            print("out of bounds")

        # print(f"SubToken starting: {token}")
    else:
        if in_row >= 1:
            # check if the previous merged token has correct offsets
            start = i + idx_offset - in_row - 1 + delta
            word = get_subtokens(in_tok, start)
            # print(f"Fixed token: {pretok_sent[out_idx - 1]}")
            # print(f"Next Token: {token}")
            # print(f"Word: {word}")
            orig_toks = tokenizer.tokenize(''.join(tok.replace('##', '') for tok in word))
            # print(f"Orig_toks: {orig_toks}")

            if len(orig_toks) > in_row + 1:
                # print("changing offsets")
                # fix the offsets
                for _ in range(len(orig_toks) - (in_row + 1)):
                    offsets_merged[out_idx - 1, 1] = token_offsets[i + idx_offset, 1]
                    idx_offset += 1
            elif len(orig_toks) < in_row + 1:
                # print("new is more than orig")
                # print(len(orig_toks), in_row + 1)
                new_delta = len(orig_toks) - (in_row + 1)
                delta += new_delta
                offsets_merged[out_idx - 1, 1] = old_offset
                idx_offset += new_delta
                # print(delta)

            # print(80 * "-")

            in_row = 0

        # handle the new token
        pretok_sent[out_idx] = token

        if in_bounds:
            offsets_merged[out_idx] = token_offsets[i + idx_offset]
        else:
            prev_end = offsets_merged[out_idx - 1, 1]
            token_len = len(token)
            offsets_merged[out_idx] = (prev_end + 1, prev_end + 1 + token_len)
        out_idx += 1
    i += 1

# Truncate arrays to actual size
pretok_sent = pretok_sent[:out_idx]
offsets_merged = offsets_merged[:out_idx]

next_starts = np.roll(offsets_merged[:, 0], -1)[:-1]
current_ends = offsets_merged[:-1, 1]
mask = next_starts > current_ends
# boolean mask checking if the offsets align or next start index is greater than previous end
# indicating space in the original text
mask = np.append(mask, False)
tokens_arr = np.array(pretok_sent)
# add spaces back at corresponding places
tokens_with_space = np.where(mask, np.char.add(pretok_sent, ' '), pretok_sent)
reconstructed_text = "".join(tokens_with_space)

In [98]:
print(*original_offsets[1:-1])
print(in_tok.astype(str).tolist())
print(tokenizer.tokenize(pred_string))
print(160 * "-")
print(pretok_sent.astype(str).tolist())
print(*offsets_merged)
print(160 * "-")
print(mask)
print(160 * "-")
reconstructed_text

(0, 1) (1, 2) (2, 3) (4, 11) (12, 15) (15, 18) (18, 21) (21, 26) (27, 32) (32, 37) (37, 38)
['#', '#', '#', 'General', 'Con', '##tri', '##but', '##ition', 'Guide', '##lines', ':']
['#', '#', '#', 'General', 'Con', '##tribution', 'Guide', '##lines', ':']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
['#', '#', '#', 'General', 'Contribution', 'Guidelines', ':']
[0 1] [1 2] [2 3] [ 4 11] [12 26] [27 37] [37 38]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[False False  True  True  True False False]
----------------------------------------------------------------------------------------------------------------------------------------------------------------


'### General Contribution Guidelines:'

In [99]:
print(orig_string)
print(80 * "-")
print(orig_string_clean)
print(80 * "-")
print(pred_string)

### General Contributition Guidelines:
--------------------------------------------------------------------------------
### General Contribution Guidelines:
--------------------------------------------------------------------------------
# # # General Contribution Guidelines :


In [94]:
tokenizer.tokenize('"Undo"')

['"', 'Un', '##do', '"']

### T5

In [11]:
from happytransformer import HappyTextToText

# Load T5 model for grammar/spelling correction
happy_tt = HappyTextToText("T5", "vennify/t5-base-grammar-correction")

# Example sentence with typos
input_text = corrupt[0]

# Correct the sentence
output = happy_tt.generate_text(f"grammar: {input_text}")
print(output.text, clean[0], sep='\n')


04/01/2025 15:56:13 - INFO - happytransformer.happy_transformer -   Using device: cuda:0
04/01/2025 15:56:14 - INFO - happytransformer.happy_transformer -   Moving model to cuda:0
04/01/2025 15:56:14 - INFO - happytransformer.happy_transformer -   Initializing a pipeline
Device set to use cuda:0


Team_number = tr[1].p.span.contents[0]
team_number = tds[1].p.span.contents[0]


### Detect typo

In [5]:
from detect_typo_model import TypoDetectionModel

In [7]:
pred_model = TypoDetectionModel()
pred_model.load_model()

Model loaded successfully
Model loaded from detect_typo_models/best_model.pt


In [16]:
for i in range(20):
    text = corrupt[i]
    print(text, clean[i], sep='\n')
    print(pred_model.predict(text))
    print(80 * '-')

team_number = tr[1].p.span.contents[0]
team_number = tds[1].p.span.contents[0]
0.47374123334884644
--------------------------------------------------------------------------------
* The internal method that handles the pointer out event from the browser.
* The internal method that handles the pointer over event from the browser.
0.5091391801834106
--------------------------------------------------------------------------------
To understand what is in the `dockercfg` field, convert the secret data to a
To understand what is in the `.dockercfg` field, convert the secret data to a
0.8158062696456909
--------------------------------------------------------------------------------
number of blocks has been removed.  The rpc calls are deprecated and will either
number of blocks has been removed.  The RPC calls are deprecated and will either
0.8373557329177856
--------------------------------------------------------------------------------
will be those of the redirected or rewriten routes w

In [18]:
skipped = 0
for i, corr in enumerate(corrupt):
    pred = pred_model.predict(corr)
    if pred < .5:
        print(pred)
        skipped += 1
        print(corr)
        print(clean[i])



0.4057482182979584
## 返回文件下载
## 文件下载
0.16591551899909973
If you are wanting to run the demos locally, just do:
If you want to run the demos locally, just do:
0.42447003722190857
.sort{ -it.estimatedRuntime }
.sort { -it.estimatedRuntime }
0.08145656436681747
Second pre-defined Redis server instance info.
Second predefined Redis server instance info.
0.43791788816452026
When running Istio-enabled services, you can use curl in one service's
When running Istio auth-enabled services, you can use curl in one service's
0.318926066160202
ASSERT_EQ(GraphNum(g), 0);
ASSERT_EQ(GraphNum(g), 0UL);
0.15489286184310913
let level8 = new Level(5, "bullet", "•", "left");
let level8 = new Level(8, "bullet", "•", "left");
0.07840577512979507
return "1";
return token;
0.3641495704650879
cd examples/guestbook-go/src
cd examples/guestbook-go/_src
0.029191849753260612
provided images for the assignments. Each student will have $100 in credit throughout the quarter. When you sign up for the first time, you al

KeyboardInterrupt: 

In [10]:
import pandas as pd

test_df = pd.read_csv(DATA_PATH + "test_prob_df.csv", dtype={0: str, 1: float})

In [11]:
pred_model.evaluate(test_df)

Test Loss: 0.0784 | Test MAE: 0.1694


{'dev_loss': 0.07842605434442387, 'mae': 0.16937337815761566}

In [34]:
from transformers import BertTokenizerFast
from neuspell.seq_modeling.helpers import merge_subtokens

tokenizer = BertTokenizerFast.from_pretrained("bert-base-cased")
tokenizer.do_basic_tokenize = True
tokenizer.tokenize_chinese_chars = False
corrupt_lines, clean_lines = get_data_from_file('test')
acc_sen, corr2corr, corr2incorr, incorr2corr, incorr2incorr = 0, 0, 0, 0, 0

idx = 4
for idx in range(len(corrupt_lines)):
    corrupt = corrupt_lines[idx]
    clean = clean_lines[idx]
    prediction = checker.correct_string(corrupt, correct_spaces=False)

    corrupt = merge_subtokens(tokenizer.tokenize(corrupt))
    clean = merge_subtokens(tokenizer.tokenize(clean))
    prediction = merge_subtokens(tokenizer.tokenize(prediction))
    # print(corrupt, clean, prediction, sep='\n')
    # print(80*'-')

    acc_sen += (prediction == clean)
    same_len = (len(corrupt) == len(clean) == len(prediction))
    if not same_len:
        for corrupt_token, clean_token, predict_token in zip(corrupt.split(), clean.split(), prediction.split()):
            if corrupt_token == clean_token and predict_token == clean_token:
                corr2corr += 1
            elif corrupt_token == clean_token and predict_token != clean_token:
                corr2incorr += 1
            elif corrupt_token != clean_token and predict_token == clean_token:
                incorr2corr += 1
            elif corrupt_token != clean_token and predict_token != clean_token:
                incorr2incorr += 1
print("Results:")
print(acc_sen, corr2corr, corr2incorr, incorr2corr, incorr2incorr)

Results:
11511 809056 45463 13702 242905


In [30]:
diff = 0
for corr, clean in zip(corrupt_lines, clean_lines):
    diff += (len(corr) == len(clean))


In [32]:
print (diff)

19070


In [33]:
len (corrupt_lines)

67886

In [6]:
from neuspell.seq_modeling.helpers import detokenize_elmo

In [20]:
detokenize_elmo(r"""team _ number =  ' tr ' " "   [ 1 ] . p . span . contents [ 0 ]""")

team
_
number
[
1
]
.
.
.
[
0
]


'team_number = \'tr\' ""[1]. p. span. contents[0]'

In [30]:
from string import punctuation


def _is_punct(inp):
    return all([i in punctuation for i in inp])
def custom_tokenizer(tokens):
    new_tokens = []
    str_ = ""
    for token in tokens:
        if _is_punct(token):
            str_ += token
        else:
            new_tokens.append(str_)
            str_ = ""
            new_tokens.append(token)
    if str_:
        new_tokens.append(str_)
    return " ".join(new_tokens)

In [21]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "ner"])

In [53]:
inp = corrupt[0]
doc = nlp(inp)
tokens = [token.text for token in doc]
tokens

team_number = tr[1].p.span.contents[0]


['team_number', '=', 'tr[1].p.span.contents[0', ']']

In [57]:
print(custom_tokenizer(tokens))

['', 'team_number', '=', 'tr[1].p.span.contents[0', ']']
 team_number = tr[1].p.span.contents[0 ]


In [56]:
doc.text

'team_number = tr[1].p.span.contents[0]'